# Аналитика данных. Стажировка «ИТ-город». Блок задач №3.

Автор: Стукалов Артем Витальевич  
Дата: 18.06.2026

## Задание 1. Загрузка данных:
- Считать данные из всех источников
- Привести данные к единому формату
- Выполнить базовую очистку
- Загрузить данные в БД

In [1]:
# импортируем библиотеки
import pandas as pd
import numpy as np
import json
import xml.etree.ElementTree as ET
from sqlalchemy import create_engine

Логируем ошибки.

In [2]:
errors = []
def log_error(source, row, err_type, msg):
    errors.append({'source': source, 'row': str(row), 'type': err_type, 'message': msg})

### Загрузим данные из всех источников.

In [3]:
# Чтение CSV
customers = pd.read_csv('customers.csv') # загрузка в customers
payments = pd.read_csv('payments.csv', sep='^') # загрузка в payments

# Чтение Excel
products = pd.read_excel('products.xlsx') # загрузка в products

# Чтение JSON
with open('orders.json', 'r') as f:
    orders_data = json.load(f)
orders = pd.DataFrame(orders_data) # загрузка в orders

# Чтение XML
tree = ET.parse('events.xml')
root = tree.getroot()

events_list = []
for event in root.findall('event'):
    event_dict = {}

    for child in event:
        event_dict[child.tag] = child.text if child.text is not None else ''
    events_list.append(event_dict)
events = pd.DataFrame(events_list)

Изучим полученные данные.

In [4]:
# Выводим основную информацию датасета customers
customers.info()
customers.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 715 entries, 0 to 714
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  715 non-null    int64 
 1   full_name    715 non-null    object
 2   email        683 non-null    object
 3   phone        562 non-null    object
 4   city         695 non-null    object
 5   created_at   715 non-null    object
dtypes: int64(1), object(5)
memory usage: 33.6+ KB


,customer_id,full_name,email,phone,city,created_at
0,1,Мишин Емельян Валерьевич,mechislav_37@hotmail.com,UNKNOWN,п. Печора,2024-05-31
1,2,Егоров Тихон Федосеевич,serafim_1986@gmail.com,UNKNOWN,п. Мостовской,2024-07-21
2,3,Зыков Вацлав Алексеевич,fharitonova@gmail.com,9749621470,ст. Луга,2024-06-15
3,4,Савельева Юлия Кузьминична,odintsovgerman@ignatev.biz,NaN,п. Воскресенск,2025-03-28
4,5,София Тимофеевна Михеева,selivan_1979@ao.ru,9206468299,г. Уренгой,2026-01-17


In [5]:
# Выводим основную информацию датасета payments
payments.info()
payments.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1012 entries, 0 to 1011
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   payment_id         1012 non-null   int64 
 1   order_id           1012 non-null   int64 
 2   payment_method     739 non-null    object
 3   amount             1012 non-null   object
 4   currency           1012 non-null   object
 5   payment_timestamp  1012 non-null   object
dtypes: int64(2), object(4)
memory usage: 47.6+ KB


,payment_id,order_id,payment_method,amount,currency,payment_timestamp
0,1,778,bank_transfer,error_amount,EUR,2025-04-14 23:23:52
1,2,941,bank_transfer,407.42,USD,2024-12-15 23:01:24
2,3,1173,paypal,1754.7,USD,2025-05-03 14:55:30
3,4,143,card,2372.04,RUB,2026-02-05 00:19:38
4,5,864,card,2366.62,EUR,2025-06-18 21:21:20


In [6]:
# Выводим основную информацию датасета products
products.info()
products.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 610 entries, 0 to 609
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    610 non-null    int64  
 1   product_name  610 non-null    object 
 2   category      610 non-null    object 
 3   price         591 non-null    float64
 4   currency      610 non-null    object 
 5   is_active     610 non-null    bool   
dtypes: bool(1), float64(1), int64(1), object(3)
memory usage: 24.6+ KB


,product_id,product_name,category,price,currency,is_active
0,1,Господь,Home,54.47,USD,False
1,2,Набор,Books,114.79,EUR,True
2,3,Вздрагивать,Electronics,49.98,RUB,False
3,4,Фонарик,Electronics,392.93,RUB,False
4,5,Фонарик,Clothing,798.13,EUR,True


In [7]:
# Выводим основную информацию датасета orders
orders.info()
orders.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1220 entries, 0 to 1219
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         1220 non-null   int64  
 1   customer_id      1165 non-null   float64
 2   product_id       1220 non-null   int64  
 3   quantity         1220 non-null   int64  
 4   unit_price       1220 non-null   float64
 5   currency         1220 non-null   object 
 6   order_timestamp  1220 non-null   object 
 7   status           1220 non-null   object 
dtypes: float64(2), int64(3), object(3)
memory usage: 76.4+ KB


,order_id,customer_id,product_id,quantity,unit_price,currency,order_timestamp,status
0,1,380.0,512,3,256.02,USD,2025-99-99,processing
1,2,467.0,117,5,218.67,EUR,2026-02-18 02:28:23,completed
2,3,470.0,391,5,89.70,EUR,2025-01-14 11:22:12,cancelled
3,4,676.0,90,3,354.03,USD,2025-05-30 03:09:57,completed
4,5,92.0,639,4,60.80,RUB,2025-01-22 15:48:14,processing


In [8]:
# Выводим основную информацию датасета events
events.info()
events.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1501 entries, 0 to 1500
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   event_id         1501 non-null   object
 1   customer_id      1501 non-null   object
 2   event_type       1501 non-null   object
 3   event_timestamp  1500 non-null   object
 4   product_id       1500 non-null   object
dtypes: object(5)
memory usage: 58.8+ KB


,event_id,customer_id,event_type,event_timestamp,product_id
0,1,483,view,2025-06-23 18:40:09,281
1,2,550,login,2025-11-30 00:01:33,20
2,3,570,login,2025-02-16 21:53:55,588
3,4,213,view,2025-07-12 08:14:08,408
4,5,202,purchase,2025-04-03 12:11:22,453


**Предварительный вывод по полученным данным:**
- Данные из пяти источников имеют хороший объём (от 715 до 1501 записей).
- Пропуски присутствуют практически во всех таблицах – в контактных данных, ценах, ключевых идентификаторах (например, customer_id в заказах), а также в событийных временных метках. Особенно критичны пропуски в полях, используемых для связей между таблицами.
- Типы данных не соответствуют смыслу колонок: даты и времена хранятся как строки, числовые суммы – как объекты (из-за ошибочных значений вроде 'error_amount'), идентификаторы в событиях – как строки, а не целые числа. Это затрудняет расчёты и объединение данных.
- Некорректные значения обнаруживаются в явном виде: нечисловые суммы, ошибочные даты (2025-99-99), строки-заглушки ('UNKNOWN' вместо отсутствующих телефонов). Такие аномалии свидетельствуют о сбоях в исходных системах или ручном вводе.
- Связность между таблицами формально есть (по customer_id, order_id, product_id), но из-за пропусков и нестандартных форматов часть записей может быть потеряна при объединении.

### Приведём данные к единому формату.

Проверим верность типов данных.

In [9]:
# создаём массив
all_data = [customers, payments, products, orders, events]
names = ['customers', 'payments', 'products', 'orders', 'events']

# пробегаемся циклом и выводим значения
for i in range(len(all_data)):
    print(names[i])
    print(all_data[i].dtypes)
    print(30*'-')

customers
customer_id     int64
full_name      object
email          object
phone          object
city           object
created_at     object
dtype: object
------------------------------
payments
payment_id            int64
order_id              int64
payment_method       object
amount               object
currency             object
payment_timestamp    object
dtype: object
------------------------------
products
product_id        int64
product_name     object
category         object
price           float64
currency         object
is_active          bool
dtype: object
------------------------------
orders
order_id             int64
customer_id        float64
product_id           int64
quantity             int64
unit_price         float64
currency            object
order_timestamp     object
status              object
dtype: object
------------------------------
events
event_id           object
customer_id        object
event_type         object
event_timestamp    object
product_id    

**После проверки корректности типа данных можно сделать следующий промежуточный вывод:**
- **Датасет customers:**
    - У столбца created_at – тип object, должен быть datetime.
    - У столбца phone – содержит значения 'UNKNOWN' (видимо, обозначение отсутствия) и пропуски NaN.
- **датасет payments:**
    - У столбца amount – тип object, но должен быть float. Также присутствует значение 'error_amount', что приведёт к ошибки при попытке преобразования данных в тип float.
    - У столбца payment_timestamp – тип object, должен быть datetime.
- **датасет orders:**
    - У столбца customer_id – тип float64, хотя должен быть int, также данный столбец содержит пропуски. 
    - У столбца order_timestamp – тип object, должна быть datetime. Присутствует явно некорректная дата: '2025-99-99' (первая строка)
- **датасет events:**
    - все столбцы имеют тип object, хотя должно быть:
        - event_id – int
        - customer_id – int
        - product_id – int
        - event_timestamp – datetime.

Исправим тип данных.

In [10]:
# customers
customers['created_at'] = pd.to_datetime(customers['created_at'], errors='coerce')
customers['phone'] = customers['phone'].replace('UNKNOWN', np.nan)  

# payments
payments['amount'] = pd.to_numeric(payments['amount'], errors='coerce')
payments['payment_timestamp'] = pd.to_datetime(payments['payment_timestamp'], errors='coerce')

# orders
orders['order_timestamp'] = pd.to_datetime(orders['order_timestamp'], format='%Y-%m-%d %H:%M:%S',  errors='coerce')

# events
events['event_timestamp'] = pd.to_datetime(events['event_timestamp'], format='%Y-%m-%d %H:%M:%S',  errors='coerce')

# Проверим результат
print("customers dtypes:\n", customers.dtypes)
print("\npayments dtypes:\n", payments.dtypes)
print("\norders dtypes:\n", orders.dtypes)
print("\nevents dtypes:\n", events.dtypes)

customers dtypes:
 customer_id             int64
full_name              object
email                  object
phone                  object
city                   object
created_at     datetime64[ns]
dtype: object

payments dtypes:
 payment_id                    int64
order_id                      int64
payment_method               object
amount                      float64
currency                     object
payment_timestamp    datetime64[ns]
dtype: object

orders dtypes:
 order_id                    int64
customer_id               float64
product_id                  int64
quantity                    int64
unit_price                float64
currency                   object
order_timestamp    datetime64[ns]
status                     object
dtype: object

events dtypes:
 event_id                   object
customer_id                object
event_type                 object
event_timestamp    datetime64[ns]
product_id                 object
dtype: object


Вероятно, тип данных object в столбцах event_id, customer_id, product_id связан с наличием в них пропусков. Таким образом, перед преобразованием типа данных в int, необходимо в начале проверить их на пропуски и в случае их наличия, избавиться от пропусков в данных столбцах.

### Выполнить базовую очистку данных.

**Проверим данные на наличие пропусков и их количество в процентном соотношении.**

In [11]:
# Выводим процент пропусков в каждом столбце всех таблиц
for i in range(len(all_data)):
    print(f"{names[i]}:\n{all_data[i].isna().mean()*100}")
    print()

customers:
customer_id     0.000000
full_name       0.000000
email           4.475524
phone          40.139860
city            2.797203
created_at      2.237762
dtype: float64

payments:
payment_id            0.000000
order_id              0.000000
payment_method       26.976285
amount                3.359684
currency              0.000000
payment_timestamp     2.075099
dtype: float64

products:
product_id      0.000000
product_name    0.000000
category        0.000000
price           3.114754
currency        0.000000
is_active       0.000000
dtype: float64

orders:
order_id           0.000000
customer_id        4.508197
product_id         0.000000
quantity           0.000000
unit_price         0.000000
currency           0.000000
order_timestamp    2.868852
status             0.000000
dtype: float64

events:
event_id           0.000000
customer_id        0.000000
event_type         0.000000
event_timestamp    3.397735
product_id         0.066622
dtype: float64



Прежде всего, нужно удалить пропуски в столбцах с идентификаторами, поскольку в них находятся уникальные ключи, которые связывают записи между таблицами. Если в таком столбце будет пропуск или синтетические данные, то нарушается целостность данных, в таком случае будет невозможно построеть корректные связи (JOIN) между таблицами. 

Сейчас это следующие столбцы c пропусками: 
- customer_id - 4.5% пропусков (таблица orders);
- product_id - 0.06% пропусков (таблица events).

In [12]:
# Удаляем строки с пропущенными идентификаторами
orders.dropna(subset=['customer_id'], inplace=True)
events.dropna(subset=['product_id'], inplace=True)

# Проверим данные
print(f'Количество пропусков в столбце customer_id (таблица orders): {orders['customer_id'].isna().sum()}')
print(f'Количество пропусков в столбце product_id (таблица events): {events['product_id'].isna().sum()}') 

Количество пропусков в столбце customer_id (таблица orders): 0
Количество пропусков в столбце product_id (таблица events): 0


Приведём к типу данных int столбцы с идентификатором.

In [13]:
# Приводим к типу данных int столбцы с идентификаторов
orders['customer_id'] = orders['customer_id'].astype('int64')
events['event_id'] = events['event_id'].astype('int64')
events['customer_id'] = events['customer_id'].astype('int64')
events['product_id'] = events['product_id'].astype('int64')

# Проверим результат
print("customers dtypes:\n", customers.dtypes)
print("\npayments dtypes:\n", payments.dtypes)
print("\norders dtypes:\n", orders.dtypes)
print("\nevents dtypes:\n", events.dtypes)

customers dtypes:
 customer_id             int64
full_name              object
email                  object
phone                  object
city                   object
created_at     datetime64[ns]
dtype: object

payments dtypes:
 payment_id                    int64
order_id                      int64
payment_method               object
amount                      float64
currency                     object
payment_timestamp    datetime64[ns]
dtype: object

orders dtypes:
 order_id                    int64
customer_id                 int64
product_id                  int64
quantity                    int64
unit_price                float64
currency                   object
order_timestamp    datetime64[ns]
status                     object
dtype: object

events dtypes:
 event_id                    int64
customer_id                 int64
event_type                 object
event_timestamp    datetime64[ns]
product_id                  int64
dtype: object


В столбцах с пропусками с числовым типов данных, а именно: 'amount' (3.35% пропусков) и 'price' (3.11% пропусков) мы ничего не будем делать с пропусками, по следующим причинам:
- Мы не можем пропуски заполнить медианой в числовых столбцах 'price' (таблица products) и 'amount' (таблица payments), поскольку в таком случае, мы создаём синтетические данные, которые могут исказить анализ;
- Мы не можем удалить пропуски в числовых столбцах: 'price' (таблица products) и 'amount' (таблица payments), а также в столбцах даты:
order_timestamp (таблица orders) и event_timestamp (таблица events), поскольку так мы теряем ценную информацию о покупателях.

In [14]:
# создаём массив
all_data = [customers, payments, products, orders, events]
names = ['customers', 'payments', 'products', 'orders', 'events']

# Выводим процент пропусков в каждом столбце всех таблиц
for name, df in zip(names, all_data):
    print(f"{name}:\n{df.isna().mean()*100}\n")

customers:
customer_id     0.000000
full_name       0.000000
email           4.475524
phone          40.139860
city            2.797203
created_at      2.237762
dtype: float64

payments:
payment_id            0.000000
order_id              0.000000
payment_method       26.976285
amount                3.359684
currency              0.000000
payment_timestamp     2.075099
dtype: float64

products:
product_id      0.000000
product_name    0.000000
category        0.000000
price           3.114754
currency        0.000000
is_active       0.000000
dtype: float64

orders:
order_id           0.000000
customer_id        0.000000
product_id         0.000000
quantity           0.000000
unit_price         0.000000
currency           0.000000
order_timestamp    2.918455
status             0.000000
dtype: float64

events:
event_id           0.000000
customer_id        0.000000
event_type         0.000000
event_timestamp    3.333333
product_id         0.000000
dtype: float64



**Проверим данные на наличие явных и неявных дубликатов.**

Проверим данные на наличие явных и неявных дубликатов. Начнём с полных дубликатов:

In [15]:
# Список датафреймов и их названий
dfs = [customers, payments, products, orders, events]
names = ['customers', 'payments', 'products', 'orders', 'events']

# Проверяем количество полных дубликатов
for df, name in zip(dfs, names):
    count = df.duplicated().sum()
    print(f"Количество полных дубликатов в датасете {name}: {count}")

Количество полных дубликатов в датасете customers: 15
Количество полных дубликатов в датасете payments: 12
Количество полных дубликатов в датасете products: 10
Количество полных дубликатов в датасете orders: 19
Количество полных дубликатов в датасете events: 0


В результате проверки на полные дубликаты мы можем увидеть, что в данных есть значения, которые полностью совпадают (полные дубликаты). Теперь нам следует их удалить.

In [16]:
# Удаляем дубликаты и выводим результат
for df, name in zip(dfs, names):
    df.drop_duplicates(inplace=True)
    print(f'Количество полных дубликатов в датасете {name}: {df.duplicated().sum()}')

Количество полных дубликатов в датасете customers: 0
Количество полных дубликатов в датасете payments: 0
Количество полных дубликатов в датасете products: 0
Количество полных дубликатов в датасете orders: 0
Количество полных дубликатов в датасете events: 0


Проверим данные на наличие неявных дубликатов.

In [17]:
# Проверяем уникальность идентификаторов
print("Дубликаты customer_id в customers:", customers['customer_id'].duplicated().sum())
print("Дубликаты order_id в orders:", orders['order_id'].duplicated().sum())
print("Дубликаты product_id в products:", products['product_id'].duplicated().sum())
print("Дубликаты payment_id в payments:", payments['payment_id'].duplicated().sum())
print("Дубликаты event_id в events:", events['event_id'].duplicated().sum())

Дубликаты customer_id в customers: 0
Дубликаты order_id в orders: 0
Дубликаты product_id в products: 0
Дубликаты payment_id в payments: 0
Дубликаты event_id в events: 0


Проверим данные на единообразие на примере столбца product_name таблицы products.

In [18]:
products['product_name'].sort_values().unique() 

array(['Аж', 'Академик', 'Аллея', 'Анализ', 'Армейский', 'Багровый',
       'Бак', 'Банда', 'Бегать', 'Беспомощный', 'Бетонный', 'Близко',
       'Боец', 'Боец   ', 'Болото', 'Ботинок', 'Бочок', 'Бровь', 'Важный',
       'Валюта', 'Вариант', 'Ведь', 'Вздрагивать', 'Вздрогнуть',
       'Виднеться', 'Висеть', 'Витрина', 'Военный', 'Возбуждение',
       'Возможно', 'Возмутиться', 'Войти', 'Волк', 'Вообще', 'Вообще   ',
       'Вперед', 'Вскакивать', 'Второй', 'Выбирать', 'Выдержать',
       'Выкинуть', 'Выражаться', 'Выражение', 'Выраженный', 'Выразить',
       'Вытаскивать', 'Головка', 'Головной', 'Горький', 'Господь',
       'Граница', 'Грустный', 'Гулять', 'Да', 'Даль', 'Дальний', 'Девка',
       'Демократия', 'Деньги', 'Добиться', 'Домашний', 'Дорогой',
       'Доставать', 'Достоинство', 'Дремать', 'Дрогнуть', 'Дружно',
       'Дурацкий', 'Дыхание', 'Дьявол', 'Дьявол   ', 'Еврейский',
       'Желание', 'Жестокий', 'Жидкий', 'Житель', 'Жить', 'Жить   ', 'За',
       'Забирать', 'Заведе

В столбце product_name датасета products есть неявные дубликаты, вызванные лишними пробелами в конце строк, что следует исправить.

In [19]:
# Удаляем пробелы
for df in [customers, payments, products, orders, events]:
    for column in df.select_dtypes(include=['object']):
        df[column] = df[column].str.strip()

In [20]:
# Проверяем результат
products['product_name'].sort_values().unique() 

array(['Аж', 'Академик', 'Аллея', 'Анализ', 'Армейский', 'Багровый',
       'Бак', 'Банда', 'Бегать', 'Беспомощный', 'Бетонный', 'Близко',
       'Боец', 'Болото', 'Ботинок', 'Бочок', 'Бровь', 'Важный', 'Валюта',
       'Вариант', 'Ведь', 'Вздрагивать', 'Вздрогнуть', 'Виднеться',
       'Висеть', 'Витрина', 'Военный', 'Возбуждение', 'Возможно',
       'Возмутиться', 'Войти', 'Волк', 'Вообще', 'Вперед', 'Вскакивать',
       'Второй', 'Выбирать', 'Выдержать', 'Выкинуть', 'Выражаться',
       'Выражение', 'Выраженный', 'Выразить', 'Вытаскивать', 'Головка',
       'Головной', 'Горький', 'Господь', 'Граница', 'Грустный', 'Гулять',
       'Да', 'Даль', 'Дальний', 'Девка', 'Демократия', 'Деньги',
       'Добиться', 'Домашний', 'Дорогой', 'Доставать', 'Достоинство',
       'Дремать', 'Дрогнуть', 'Дружно', 'Дурацкий', 'Дыхание', 'Дьявол',
       'Еврейский', 'Желание', 'Жестокий', 'Жидкий', 'Житель', 'Жить',
       'За', 'Забирать', 'Заведение', 'Задержать', 'Заложить', 'Запеть',
       'Заплак

**Предварительный вывод по полученным данным:**
- **Прочитаны все источники с учётом особенностей форматов.**
- **Выявлены нессответствия типовы данных:**
    -  Столбцы с датой и временем, а именно: created_at (таблица customers), payment_timestamp (таблица payments), order_timestamp (таблица orders), event_timestamp (таблица events) имели неверный тип данных object, в дальнейшем приведены к верному типу данных datetime64
    -  Столбец amount приведён к float64, некорректные значения стали NaN.
    -  Идентификатор customer_id в таблице orders был float64 и Идентификаторы event_id, customer_id, product_id в таблице events были object, после удаления пропусков приведены к int64.
- **Обработка пропусков в данных:**
    - Обработаны критически важные пропуски в идентификаторах:
        - customer_id (4.5%) датасета orders - удалены строки с пропущенным id клиента.
        - events.product_id (0.06%) датасета events – удалены строки с пропущенным id продукта.
    - В остальных столбцах пропуски оставлены без заполнения по следующим причинам:
        -  Заполнение медианой/средним пропусков в числовых столбцах price, amount внесло бы синтетические искажения.
        -  Удаление строк в числовых столбцах (price, amount) и столбцах даты и времени (order_timestamp, event_timestamp) привело бы к потере полезной информации.
- **Обработка дубликатов:**
    - Полные дубликаты:
        - Обнаружены в датасетах customers (15), payments (12), products (10) и orders (19). В events полных дубликатов не найдено.
        - Удалены все полные дубликаты.
    - Уникальность идентификаторов:
        - Проверка по первичным ключам (customer_id, order_id, product_id, payment_id, event_id) показала отсутствие дубликатов во всех таблицах. Это гарантирует корректность связей между таблицами.
    - Неявные дубликаты и единообразие строк:
        - В столбце product_name датасета products выявлены неявные дубликаты, вызванные лишними пробелами в конце строк
        - У всех значений строковых столбцов удалены лишние пробелы в начале и конце, также данные приведены к единому нижнему регистру.

## Загрузим данные в БД

In [21]:
# Создаём файл базы данных dwh.db в текущей папке
engine = create_engine('sqlite:///dwh.db')

# Загружаем каждый датафрейм в отдельную таблицу
customers.to_sql('customers', engine, if_exists='replace', index=False)
payments.to_sql('payments', engine, if_exists='replace', index=False)
products.to_sql('products', engine, if_exists='replace', index=False)
orders.to_sql('orders', engine, if_exists='replace', index=False)
events.to_sql('events', engine, if_exists='replace', index=False)

print("Данные успешно загружены в файл dwh.db")

Данные успешно загружены в файл dwh.db


## Задание 2. Data Quality: Необходимо обработать ошибки, описать принятые решения и логировать проблемные записи.

In [23]:
# Список таблиц и столбцов для проверки
checks = [
    ('customers', customers, ['created_at']),
    ('payments', payments, ['amount', 'payment_timestamp']),
    ('products', products, ['price']),
    ('orders', orders, ['order_timestamp']),
    ('events', events, ['event_timestamp'])
]

In [24]:
# Проверяем все таблицы и столбцы, где могли быть ошибки.
for source, df, columns in checks:
    for column in columns:
        if column in df.columns:
            bad = df[column].isna()
            if bad.any():
                for idx in df[bad].index:
                    log_error(source, df.loc[idx].to_dict(), 'conversion', f"Ошибка в колонке {column} (значение отсутствует или неверно)")

In [25]:
# Проверяем дубликаты
for source, df in [('customers', customers), ('payments', payments), ('products', products), ('orders', orders), ('events', events)]:
    dup = df.duplicated().sum()
    if dup:
        log_error(source, {}, 'duplicate', f"Найдено {dup} дубликатов")

In [26]:
#Сохраняем ошибки в CSV
if errors:
    pd.DataFrame(errors).to_csv('error_log.csv', index=False)
    print(f"Ошибки сохранены в error_log.csv (всего {len(errors)})")
else:
    print("Ошибок нет!")

Ошибки сохранены в error_log.csv (всего 170)
